# IEEE-CIS Fraud Detection — Exploratory Data Analysis

**Purpose:** Pre-training gate. Validate data structure, confirm no leakage, establish the modelling baseline before Day 3.

**Sections:**
1. Load & join
2. Class balance
3. TransactionDT temporal structure + split boundaries
4. Feature distributions (Amount, ProductCD, card types)
5. Missingness patterns (V / C / D / M feature groups)
6. Leakage scan
7. Key findings summary

**Rule:** Nothing in this notebook trains a model or touches the test split.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (13, 4.5)})
sns.set_theme(style="whitegrid", palette="muted")

RAW = Path("../data/raw")
assert RAW.exists(), "data/raw/ not found — run the data download step first"
print("Setup OK")

## 1. Load & Join

In [ ]:
txn = pd.read_csv(RAW / "train_transaction.csv")
idn = pd.read_csv(RAW / "train_identity.csv")
df = txn.merge(idn, on="TransactionID", how="left")

print(f"train_transaction : {txn.shape[0]:>7,} rows  ×  {txn.shape[1]:>3} cols")
print(f"train_identity    : {idn.shape[0]:>7,} rows  ×  {idn.shape[1]:>3} cols")
print(f"joined (left)     : {df.shape[0]:>7,} rows  ×  {df.shape[1]:>3} cols")
print(
    f"Identity join rate: {idn.shape[0] / txn.shape[0] * 100:.1f}%  "
    f"({txn.shape[0] - idn.shape[0]:,} transactions have no identity record)"
)

## 2. Class Balance

In [ ]:
fraud_n = df["isFraud"].sum()
legit_n = len(df) - fraud_n
fraud_pct = fraud_n / len(df) * 100

print(f"Fraud     : {fraud_n:>7,}  ({fraud_pct:.2f}%)")
print(f"Legitimate: {legit_n:>7,}  ({100 - fraud_pct:.2f}%)")
print(f"Imbalance ratio: 1 : {legit_n / fraud_n:.0f}")

fig, axes = plt.subplots(1, 2)

# Count bar
axes[0].bar(["Legitimate", "Fraud"], [legit_n, fraud_n], color=["steelblue", "tomato"])
axes[0].set_title("Transaction counts")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

# Percentage pie
axes[1].pie(
    [legit_n, fraud_n],
    labels=["Legitimate", "Fraud"],
    colors=["steelblue", "tomato"],
    autopct="%1.2f%%",
    startangle=90,
)
axes[1].set_title("Class proportions")

plt.suptitle("Class balance — IEEE-CIS training set", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Fraud rate by ProductCD — relevant for subgroup evaluation in the model card
prod_stats = (
    df.groupby("ProductCD")["isFraud"]
    .agg(total="count", fraud="sum")
    .assign(fraud_rate=lambda x: x["fraud"] / x["total"] * 100)
    .sort_values("fraud_rate", ascending=False)
)
print("Fraud rate by ProductCD:")
print(prod_stats.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
prod_stats["fraud_rate"].plot(kind="bar", ax=ax, color="tomato", edgecolor="white")
ax.axhline(fraud_pct, color="navy", linestyle="--", label=f"Overall {fraud_pct:.2f}%")
ax.set_title("Fraud rate by ProductCD")
ax.set_ylabel("Fraud rate (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Temporal Structure

In [ ]:
dt = df["TransactionDT"]
dt_range_days = (dt.max() - dt.min()) / 86_400

# Compute the 70/85 split boundaries used in temporal_split.py
dt_sorted = dt.sort_values()
n = len(dt_sorted)
train_end = int(dt_sorted.iloc[int(n * 0.70)])
val_end = int(dt_sorted.iloc[int(n * 0.85)])

print(f"TransactionDT range : {dt.min():,} → {dt.max():,}")
print(f"Span                : {dt_range_days:.1f} days (~{dt_range_days/30:.1f} months)")
print(f"train_end_dt        : {train_end:,}  (day {(train_end - dt.min()) / 86400:.0f})")
print(f"val_end_dt          : {val_end:,}  (day {(val_end - dt.min()) / 86400:.0f})")
print("\nSplit sizes:")
print(f"  Train : {(dt < train_end).sum():>7,}  ({(dt < train_end).sum()/n*100:.1f}%)")
print(
    f"  Val   : {((dt >= train_end) & (dt < val_end)).sum():>7,}  ({((dt >= train_end) & (dt < val_end)).sum()/n*100:.1f}%)"
)
print(f"  Test  : {(dt >= val_end).sum():>7,}  ({(dt >= val_end).sum()/n*100:.1f}%)")

In [ ]:
# Transaction volume and fraud rate over time (100 equal-width time bins)
N_BINS = 100
df["dt_bin"] = pd.cut(df["TransactionDT"], bins=N_BINS)
binned = (
    df.groupby("dt_bin", observed=True)
    .agg(count=("isFraud", "count"), fraud_rate=("isFraud", "mean"))
    .reset_index()
)
bin_midpoints = binned["dt_bin"].apply(lambda x: x.mid).astype(float)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Volume
split_colors = []
for mid in bin_midpoints:
    if mid < train_end:
        split_colors.append("steelblue")
    elif mid < val_end:
        split_colors.append("darkorange")
    else:
        split_colors.append("tomato")

ax1.bar(range(N_BINS), binned["count"], color=split_colors, width=1)
ax1.set_ylabel("Transaction count")
ax1.set_title("Transaction volume over time")
from matplotlib.patches import Patch

ax1.legend(
    handles=[
        Patch(color="steelblue", label="Train (70%)"),
        Patch(color="darkorange", label="Val (15%)"),
        Patch(color="tomato", label="Test (15%)"),
    ],
    loc="upper right",
)

# Fraud rate
ax2.plot(range(N_BINS), binned["fraud_rate"] * 100, color="crimson", linewidth=1.5)
ax2.axhline(fraud_pct, color="navy", linestyle="--", alpha=0.6, label=f"Mean {fraud_pct:.2f}%")
ax2.axvline(int(N_BINS * 0.70), color="black", linestyle=":", alpha=0.7, label="Train/Val split")
ax2.axvline(int(N_BINS * 0.85), color="gray", linestyle=":", alpha=0.7, label="Val/Test split")
ax2.set_ylabel("Fraud rate (%)")
ax2.set_xlabel("Time bins (earliest → latest)")
ax2.set_title("Fraud rate over time")
ax2.legend()

plt.tight_layout()
plt.show()
df.drop(columns=["dt_bin"], inplace=True)

## 4. Feature Distributions

In [ ]:
# TransactionAmt: log-scale, split by fraud label
fraud_amt = df.loc[df["isFraud"] == 1, "TransactionAmt"]
legit_amt = df.loc[df["isFraud"] == 0, "TransactionAmt"]

print("TransactionAmt summary by class:")
print(df.groupby("isFraud")["TransactionAmt"].describe().round(2).to_string())

fig, axes = plt.subplots(1, 2)

# Log-scale histogram
bins = np.logspace(
    np.log10(df["TransactionAmt"].min() + 0.01), np.log10(df["TransactionAmt"].max()), 60
)
axes[0].hist(legit_amt, bins=bins, alpha=0.6, label="Legitimate", color="steelblue", density=True)
axes[0].hist(fraud_amt, bins=bins, alpha=0.6, label="Fraud", color="tomato", density=True)
axes[0].set_xscale("log")
axes[0].set_title("TransactionAmt distribution (log scale)")
axes[0].set_xlabel("Amount (log scale)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Boxplot
df.boxplot(
    column="TransactionAmt",
    by="isFraud",
    ax=axes[1],
    showfliers=False,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue"),
)
axes[1].set_title("Amount by fraud label (no outliers)")
axes[1].set_xlabel("isFraud")
axes[1].set_ylabel("TransactionAmt")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Card type distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ["card4", "card6", "ProductCD"], strict=False):
    vc = df[col].value_counts()
    vc.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"{col} distribution")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

plt.suptitle("Categorical feature distributions", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Missingness Patterns

In [ ]:
# High-level null rate by feature group
def null_rate_summary(data, prefix, label):
    cols = [c for c in data.columns if c.startswith(prefix)]
    if not cols:
        return None
    rates = data[cols].isna().mean() * 100
    return pd.Series(
        {
            "feature_group": label,
            "n_features": len(cols),
            "null_rate_min": rates.min(),
            "null_rate_mean": rates.mean(),
            "null_rate_max": rates.max(),
            "fully_null_pct": (rates == 100).mean() * 100,
        }
    )


groups = [
    null_rate_summary(df, "V", "V-features (device/identity)"),
    null_rate_summary(df, "C", "C-features (counts)"),
    null_rate_summary(df, "D", "D-features (time deltas)"),
    null_rate_summary(df, "M", "M-features (match flags)"),
]
summary = pd.DataFrame([g for g in groups if g is not None]).set_index("feature_group")
print("Null rate summary by feature group:")
print(summary.round(1).to_string())

In [ ]:
# V-feature missingness distribution (the 339 identity/device features)
v_cols = [c for c in df.columns if c.startswith("V")]
v_null = df[v_cols].isna().mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(v_null, bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("V-feature null-rate distribution")
axes[0].set_xlabel("Null rate (%)")
axes[0].set_ylabel("Number of features")

# Cumulative: what fraction of V-features exceed a given null threshold?
thresholds = np.linspace(0, 100, 200)
cum_frac = [(v_null > t).mean() for t in thresholds]
axes[1].plot(thresholds, [f * 100 for f in cum_frac], color="tomato")
axes[1].set_title("V-features: % with null rate > threshold")
axes[1].set_xlabel("Null rate threshold (%)")
axes[1].set_ylabel("% of V-features above threshold")
axes[1].axvline(50, color="gray", linestyle="--", alpha=0.7, label=">50% null")
axes[1].legend()

plt.suptitle("V-feature missingness (n=339)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"V-features with >50% null: {(v_null > 50).sum()} / {len(v_cols)}")
print(f"V-features with >90% null: {(v_null > 90).sum()} / {len(v_cols)}")

In [ ]:
# D and C feature null rates (time-delta and counting features)
d_cols = [c for c in df.columns if c.startswith("D")]
c_cols = [c for c in df.columns if c.startswith("C")]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, cols, label in [
    (axes[0], d_cols, "D-features (time deltas)"),
    (axes[1], c_cols, "C-features (counts)"),
]:
    rates = df[cols].isna().mean() * 100
    ax.bar(cols, rates, color="darkorange", edgecolor="white")
    ax.set_title(f"{label} null rates")
    ax.set_ylabel("Null rate (%)")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.axhline(50, color="red", linestyle="--", alpha=0.6, label="50% threshold")
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Leakage Scan

Three checks:
1. **TransactionID** — sequential IDs may correlate with fraud rate if fraud is time-concentrated. Should not be used as a feature.
2. **Top raw feature correlations** — flag any feature with |correlation| > 0.3 with isFraud. In a legitimate feature set this should be rare; anything above 0.5 warrants investigation.
3. **D-feature temporal check** — D-features represent time deltas computed by Vesta. Verify they contain plausible values and no negative deltas (which would indicate future information).

In [ ]:
# Check 1: TransactionID vs fraud rate
# If fraud is time-concentrated and IDs are sequential, ID will correlate with label
corr_id = df["TransactionID"].corr(df["isFraud"])
print(f"TransactionID ↔ isFraud correlation: {corr_id:.4f}")
if abs(corr_id) > 0.05:
    print("  ⚠  Non-trivial correlation detected.")
    print("  TransactionID must NOT be included as a model feature.")
    print("  This reflects temporal clustering of fraud, not a usable signal.")
else:
    print("  ✓  Low correlation — ID is not a leakage vector here.")

# Visualise: fraud rate in rolling 5k transaction windows by ID order
id_sorted = df.sort_values("TransactionID").reset_index(drop=True)
rolling_fraud = id_sorted["isFraud"].rolling(5000, min_periods=100).mean() * 100

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(rolling_fraud.index, rolling_fraud.values, color="tomato", linewidth=1)
ax.axhline(fraud_pct, color="navy", linestyle="--", alpha=0.7, label=f"Mean {fraud_pct:.2f}%")
ax.set_title(
    "Rolling fraud rate by TransactionID order (5k window) — should NOT be used as feature"
)
ax.set_xlabel("Row index (sorted by TransactionID)")
ax.set_ylabel("Fraud rate (%)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Check 2: Top raw feature correlations with isFraud
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["isFraud", "TransactionID"]]

print("Computing correlations (this takes ~30s on full dataset)...")
corrs = df[numeric_cols].corrwith(df["isFraud"]).dropna().abs().sort_values(ascending=False)

top20 = corrs.head(20)
high_corr = corrs[corrs > 0.3]

print(f"\nFeatures with |correlation| > 0.30 with isFraud: {len(high_corr)}")
if len(high_corr) > 0:
    print(high_corr.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
top20.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.axhline(0.3, color="red", linestyle="--", alpha=0.7, label="0.30 leakage threshold")
ax.set_title("Top 20 raw feature correlations with isFraud (|Pearson r|)")
ax.set_ylabel("|Correlation|")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Check 3: D-feature temporal validity
# D-features should be non-negative (they are time DELTAS — days since last event).
# Negative values would mean the referenced event is in the FUTURE → leakage.
d_cols = [c for c in df.columns if c.startswith("D")]

print("D-feature sanity check (negative values = future data = leakage risk):")
issues = []
for col in d_cols:
    neg_count = (df[col] < 0).sum()
    neg_pct = neg_count / df[col].notna().sum() * 100 if df[col].notna().sum() > 0 else 0
    null_pct = df[col].isna().mean() * 100
    status = "⚠ NEGATIVE VALUES" if neg_count > 0 else "✓"
    print(f"  {col:4s}: {status:25s}  neg={neg_count:>5,} ({neg_pct:.1f}%)  null={null_pct:.1f}%")
    if neg_count > 0:
        issues.append(col)

if issues:
    print(f"\n⚠  {len(issues)} D-features contain negative values: {issues}")
    print("   These may represent data quality issues in Vesta's delta computation,")
    print("   not forward leakage. Clip to 0 in feature_engineering.py before training.")
else:
    print("\n✓  All D-features are non-negative — no temporal leakage detected.")

## 7. Key Findings

Run the cell below after completing all sections. It prints a structured summary for the Day 2 gate.

In [ ]:
print("=" * 60)
print("DAY 2 EDA GATE — KEY FINDINGS")
print("=" * 60)

print(f"""
DATASET
  Transactions  : {len(txn):,} rows × {txn.shape[1]} columns
  Identity rows : {len(idn):,} ({len(idn)/len(txn)*100:.1f}% have identity data)
  Joined shape  : {df.shape[0]:,} rows × {df.shape[1]} columns

CLASS BALANCE
  Fraud rate    : {fraud_pct:.2f}%  ({fraud_n:,} / {len(df):,})
  Imbalance     : 1 : {legit_n//fraud_n}  → requires class_weight or scale_pos_weight in LightGBM

TEMPORAL STRUCTURE
  Span          : {dt_range_days:.0f} days
  train_end_dt  : {train_end:,}  (row {int(n*0.70):,})
  val_end_dt    : {val_end:,}  (row {int(n*0.85):,})
  Split sizes   : 70% / 15% / 15%

MISSINGNESS
  V-features >50% null : {(df[[c for c in df.columns if c.startswith('V')]].isna().mean() > 0.5).sum()} / 339
  → LightGBM handles NaN natively; do NOT impute V-features before training
  → Do NOT drop V-features — high missingness is informative (card type = no device data)

LEAKAGE SCAN
  TransactionID corr  : {corr_id:.4f}  → EXCLUDE from features
  Features |corr|>0.3 : {len(high_corr)}
  D-feature negatives : {len(issues)} features affected  → clip to 0 before training

DAY 3 GATE
  ✓  Data present and correct shape
  ✓  Fraud rate {fraud_pct:.2f}% (expected ~3.5%)
  ✓  Temporal structure validated — split boundaries set
  ✓  TransactionID excluded from features
  {'✓  No high-correlation leakage features' if len(high_corr) == 0 else f'⚠  {len(high_corr)} features with |corr|>0.3 — review before training'}
  {'✓  D-features clean' if len(issues) == 0 else f'⚠  Clip {len(issues)} D-features to 0 min in feature_engineering.py'}
""")

print("=" * 60)
print("READY FOR DAY 3 — model training")
print("=" * 60)